In [0]:
%python
spark.range(1).show()

+---+
| id|
+---+
|  0|
+---+



In [0]:
%sql
SELECT current_user() AS usuario,
       current_catalog() AS catalogo,
       current_schema() AS esquema

usuario,catalogo,esquema
05valarist@gmail.com,workspace,default


In [0]:
%sql
SHOW CATALOGS

catalog
samples
system
workspace


1. Creación del entorno

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS analitica

In [0]:
%sql
USE CATALOG analitica

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS analitica.`data`;


In [0]:
%sql
USE SCHEMA `data`

In [0]:
%sql
SHOW SCHEMAS IN analitica

databaseName
data
default
information_schema


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS analitica.data.landing

In [0]:
%sql
SHOW VOLUMES IN analitica.data

database,volume_name
data,landing


In [0]:
%sql
CREATE OR REPLACE TABLE tabla_simple (
  id INT,
  nombre STRING
)
USING DELTA

In [0]:
%sql
INSERT INTO tabla_simple VALUES (1, 'Ana'), (2, 'Luis'), (3, 'María')

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM tabla_simple ORDER BY id

id,nombre
1,Ana
2,Luis
3,María


In [0]:
%python
from pyspark.sql import functions as F

In [0]:
ruta = "/Volumes/analitica/data/landing"

In [0]:
df_inmuebles = spark.read.csv(
    ruta + "/data_demo.csv", header=True, inferSchema=True
)

display(df_inmuebles.limit(10))

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0
3.0,3.0,198.0,3116290.0,3,306.3557294709182,0
4.0,2.0,245.0,1202560.0,3,24.824464122611424,0
2.0,2.0,66.0,419282.0,5,24.0,0
2.0,1.0,114.0,1830260.0,1,251.39533835163672,0
2.0,1.0,56.0,274005.0,5,93.0,0


In [0]:
df_inmuebles.printSchema()
print("Inmuebles:", df_inmuebles.count())

root
 |-- rooms: double (nullable = true)
 |-- garages: double (nullable = true)
 |-- useful_area: double (nullable = true)
 |-- value: double (nullable = true)
 |-- interior_quality: integer (nullable = true)
 |-- time_on_market: double (nullable = true)
 |-- has_outlier: integer (nullable = true)

Inmuebles: 8969


In [0]:
 
path_data_demo = "/Volumes/analitica/data/landing/data_demo.csv"

df = spark.read.csv(
    path=path_data_demo,
    header=True,
    inferSchema=True
)


In [0]:
display(df.limit(5))

rooms,garages,useful_area,value,interior_quality,time_on_market,has_outlier
3.0,1.0,105.0,1038640.0,2,31.7870789438591,0
3.0,2.0,76.0,606405.0,4,209.46884172046086,0
3.0,1.0,123.0,1534500.0,1,38.0,0
3.0,2.0,180.0,1131950.0,5,154.0,0
3.0,1.0,67.0,452672.0,4,15.0,0


In [0]:
df_nutrientes = spark.read.csv(
    ruta + "/nutrients_csvfile.csv", header=True, inferSchema=False
)

display(df_nutrientes.select("Food", "Calories", "Fat", "Category").limit(10))

Food,Calories,Fat,Category
Cows' milk,660,40,Dairy products
Milk skim,360,t,Dairy products
Buttermilk,127,5,Dairy products
"Evaporated, undiluted",345,20,Dairy products
Fortified milk,"1,373",42,Dairy products
Powdered milk,515,28,Dairy products
"skim, instant",290,t,Dairy products
"skim, non-instant",290,t,Dairy products
Goats' milk,165,10,Dairy products
(1/2 cup ice cream),690,24,Dairy products


In [0]:
df_nutrientes.printSchema()
print("Alimentos:", df_nutrientes.count())

root
 |-- Food: string (nullable = true)
 |-- Measure: string (nullable = true)
 |-- Grams: string (nullable = true)
 |-- Calories: string (nullable = true)
 |-- Protein: string (nullable = true)
 |-- Fat: string (nullable = true)
 |-- Sat.Fat: string (nullable = true)
 |-- Fiber: string (nullable = true)
 |-- Carbs: string (nullable = true)
 |-- Category: string (nullable = true)

Alimentos: 335


In [0]:
%sql
SHOW TABLES IN analitica.data

database,tableName,isTemporary
data,tabla_simple,false


In [0]:
df_pedidos = spark.read.csv(
    ruta + "/spark_intro.csv", header=True,
    schema="""order_id INT, order_date DATE, country STRING, product STRING,
              category STRING, units_sold INT, unit_price DOUBLE""",
    dateFormat="yyyy-MM-dd", mode="FAILFAST"
)

display(df_pedidos)

order_id,order_date,country,product,category,units_sold,unit_price
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0
6,2024-01-08,France,Mouse,Accessories,10,23.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0
9,2024-01-12,France,Monitor,Electronics,1,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0


In [0]:
display(
    df_pedidos.select("order_id", "country", "product", "units_sold")
    .filter(F.col("country") == "Spain")
)

order_id,country,product,units_sold
1,Spain,Laptop,2
2,Spain,Mouse,5
5,Spain,Monitor,2
8,Spain,Keyboard,4


In [0]:
df_importes = df_pedidos.withColumn(
    "importe", F.col("units_sold") * F.col("unit_price")
)

display(df_importes)

order_id,order_date,country,product,category,units_sold,unit_price,importe
1,2024-01-02,Spain,Laptop,Electronics,2,1200.0,2400.0
2,2024-01-03,Spain,Mouse,Accessories,5,25.0,125.0
3,2024-01-05,France,Laptop,Electronics,1,1150.0,1150.0
4,2024-01-06,Germany,Keyboard,Accessories,3,45.0,135.0
5,2024-01-07,Spain,Monitor,Electronics,2,300.0,600.0
6,2024-01-08,France,Mouse,Accessories,10,23.0,230.0
7,2024-01-10,Germany,Laptop,Electronics,1,1300.0,1300.0
8,2024-01-11,Spain,Keyboard,Accessories,4,42.0,168.0
9,2024-01-12,France,Monitor,Electronics,1,280.0,280.0
10,2024-01-14,Germany,Mouse,Accessories,6,27.0,162.0


In [0]:
resumen_pais = df_importes.groupBy("country").agg(
    F.sum("units_sold").alias("unidades"),
    F.round(F.sum("importe"), 2).alias("importe_total")
)

display(resumen_pais.orderBy(F.desc("importe_total")))

country,unidades,importe_total
Spain,13,3293.0
France,12,1660.0
Germany,10,1597.0


In [0]:
df_productos = spark.read.csv(
    ruta + "/dim_spark_intro.csv", header=True,
    schema="product STRING, product_family STRING, supplier STRING",
    mode="FAILFAST"
)

display(df_productos)

product,product_family,supplier
Laptop,Computers,TechCorp
Mouse,Peripherals,InputMakers
Keyboard,Peripherals,InputMakers
Monitor,Displays,ScreenWorld


In [0]:
display(df_productos.groupBy("product").count().filter("count > 1"))

df_enriquecido = df_importes.join(df_productos, "product", "left")

display(df_enriquecido)
print("Pedidos antes:", df_pedidos.count())
print("Pedidos después:", df_enriquecido.count())
display(df_pedidos.join(df_productos, "product", "left_anti"))

assert df_enriquecido.count() == df_pedidos.count(), "Revisar la llave de productos."

product,count


product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200.0,2400.0,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25.0,125.0,Peripherals,InputMakers
Laptop,3,2024-01-05,France,Electronics,1,1150.0,1150.0,Computers,TechCorp
Keyboard,4,2024-01-06,Germany,Accessories,3,45.0,135.0,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300.0,600.0,Displays,ScreenWorld
Mouse,6,2024-01-08,France,Accessories,10,23.0,230.0,Peripherals,InputMakers
Laptop,7,2024-01-10,Germany,Electronics,1,1300.0,1300.0,Computers,TechCorp
Keyboard,8,2024-01-11,Spain,Accessories,4,42.0,168.0,Peripherals,InputMakers
Monitor,9,2024-01-12,France,Electronics,1,280.0,280.0,Displays,ScreenWorld
Mouse,10,2024-01-14,Germany,Accessories,6,27.0,162.0,Peripherals,InputMakers


Pedidos antes: 10
Pedidos después: 10


product,order_id,order_date,country,category,units_sold,unit_price


In [0]:
df_enriquecido.write.format("delta").mode("overwrite").saveAsTable("pedidos_enriquecidos")

In [0]:
%sql
SELECT product, product_family, supplier, importe FROM pedidos_enriquecidos ORDER BY order_id

product,product_family,supplier,importe
Laptop,Computers,TechCorp,2400.0
Mouse,Peripherals,InputMakers,125.0
Laptop,Computers,TechCorp,1150.0
Keyboard,Peripherals,InputMakers,135.0
Monitor,Displays,ScreenWorld,600.0
Mouse,Peripherals,InputMakers,230.0
Laptop,Computers,TechCorp,1300.0
Keyboard,Peripherals,InputMakers,168.0
Monitor,Displays,ScreenWorld,280.0
Mouse,Peripherals,InputMakers,162.0


In [0]:
df_ventas = spark.read.csv(
    ruta + "/ventas_2025.csv", header=True,
    schema="id INT, fecha DATE, producto STRING, cantidad INT, precio DOUBLE",
    dateFormat="yyyy-MM-dd", mode="FAILFAST"
)

display(df_ventas.limit(10))
df_ventas.printSchema()
print("Registros de venta:", df_ventas.count())

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99
2,2025-01-03,Webcam,1,39.99
3,2025-01-03,Auriculares,18,29.99
4,2025-01-04,Webcam,10,39.99
5,2025-01-04,Mouse,2,15.5
6,2025-01-05,Altavoces,13,45.0
7,2025-01-08,Teclado,11,25.99
8,2025-01-10,Auriculares,5,29.99
9,2025-01-11,Monitor,2,199.99
10,2025-01-12,Microfono,20,59.99


root
 |-- id: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio: double (nullable = true)

Registros de venta: 100


In [0]:
display(df_ventas.groupBy("id").count().filter("count > 1"))

df_invalidos = df_ventas.filter(
    F.col("id").isNull() | F.col("fecha").isNull() |
    F.col("producto").isNull() | F.col("cantidad").isNull() |
    F.col("precio").isNull() | F.isnan("precio") |
    (F.col("cantidad") <= 0) | (F.col("precio") < 0)
)
display(df_invalidos)
assert df_invalidos.count() == 0, "Revisar los registros antes de continuar."

id,count


id,fecha,producto,cantidad,precio


In [0]:
df_ventas.write.format("delta").mode("overwrite").saveAsTable("ventas")

In [0]:
%sql 
DESCRIBE DETAIL ventas

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,52ac4698-d013-44d8-9948-a169f7000599,workspace.default.ventas,null,,2026-10-17T20:48:09.035Z,2026-10-17T20:48:18.000Z,List(),List(),1,2392,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
SELECT producto,
       SUM(cantidad) AS unidades,
       ROUND(SUM(cantidad * precio), 2) AS importe_total
FROM ventas
GROUP BY producto
ORDER BY importe_total DESC

producto,unidades,importe_total
Monitor,64,12799.36
Impresora,85,7641.5
Altavoces,137,6165.0
Disco Duro,77,6159.23
Microfono,93,5579.07
Webcam,137,5478.63
Teclado,130,3378.7
Auriculares,92,2759.08
Mouse,88,1364.0
USB,99,989.01


Databricks visualization. Run in Databricks to view.

In [0]:
pais_elegido = "Spain"

display(df_enriquecido.filter(F.col("country") == pais_elegido))

product,order_id,order_date,country,category,units_sold,unit_price,importe,product_family,supplier
Laptop,1,2024-01-02,Spain,Electronics,2,1200.0,2400.0,Computers,TechCorp
Mouse,2,2024-01-03,Spain,Accessories,5,25.0,125.0,Peripherals,InputMakers
Monitor,5,2024-01-07,Spain,Electronics,2,300.0,600.0,Displays,ScreenWorld
Keyboard,8,2024-01-11,Spain,Accessories,4,42.0,168.0,Peripherals,InputMakers


In [0]:
%sql
CREATE OR REPLACE TABLE ventas_delta_demo USING DELTA AS SELECT * FROM ventas

num_affected_rows,num_inserted_rows


In [0]:
historial = spark.sql("DESCRIBE HISTORY ventas_delta_demo")
version_inicial = historial.orderBy(F.desc("version")).first()["version"]

print("Versión de referencia:", version_inicial)

Versión de referencia: 0


In [0]:
%sql
UPDATE ventas_delta_demo SET precio = ROUND(precio + 1, 2) WHERE id = 1

num_affected_rows
1


In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id = 1

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,26.99


In [0]:
%sql
INSERT INTO ventas_delta_demo VALUES (1001, DATE '2025-01-31', 'Producto demo', 2, 10.0)

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

registros
101


In [0]:
%sql
DELETE FROM ventas_delta_demo WHERE id = 1001

num_affected_rows
1


In [0]:
%sql
SELECT COUNT(*) AS registros FROM ventas_delta_demo

registros
100


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW cambios_ventas AS
SELECT * FROM VALUES
  (1, DATE '2025-01-03', 'Teclado', 20, CAST(27.99 AS DOUBLE)),
  (1001, DATE '2025-01-31', 'Producto demo', 2, CAST(10.0 AS DOUBLE))
AS cambios(id, fecha, producto, cantidad, precio)

In [0]:
%sql
SELECT * FROM cambios_ventas

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


In [0]:
%sql
MERGE INTO ventas_delta_demo AS destino
USING cambios_ventas AS fuente
ON destino.id = fuente.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM ventas_delta_demo WHERE id IN (1, 1001) ORDER BY id

id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99
1001,2025-01-31,Producto demo,2,10.0


In [0]:
%sql
DESCRIBE HISTORY ventas_delta_demo

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-10-17T20:57:27.000Z,71239706797692,05valarist@gmail.com,MERGE,"Map(predicate -> [""(id#13498 = id#13483)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(2098451386685226),c561ccbc-dd8c-4eef-839a-3b3e21cce0ca,1017-203441-69s6igld-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 3293, numTargetBytesRemoved -> 1660, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 4042, materializeSourceTimeMs -> 336, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1983, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 1, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1641)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
3,2026-10-17T20:55:04.000Z,71239706797692,05valarist@gmail.com,DELETE,"Map(predicate -> [""(id#13187 = 1001)""])",null,List(2098451386685226),b32d4dfb-f470-4290-a471-79f342290bdd,1017-203441-69s6igld-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1663, numCopiedRows -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1652, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 1477, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 174)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
2,2026-10-17T20:52:37.000Z,71239706797692,05valarist@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2098451386685226),d2727b9b-6c96-441a-b783-70f0d2e9d25f,1017-203441-69s6igld-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 1663)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
1,2026-10-17T20:52:18.000Z,71239706797692,05valarist@gmail.com,UPDATE,"Map(predicate -> [""(id#12385 = 1)""])",null,List(2098451386685226),21d85e53-f158-4d94-b493-67b152c8ec0a,1017-203441-69s6igld-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 4432, numDeletionVectorsUpdated -> 0, scanTimeMs -> 2209, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1660, rewriteTimeMs -> 2184)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13
0,2026-10-17T20:51:56.000Z,71239706797692,05valarist@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2098451386685226),c172f2b4-0643-46ca-8555-852ef0305875,1017-203441-69s6igld-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 100, numOutputBytes -> 2392)",null,Databricks-Runtime/19.9.x-aarch64-photon-scala2.13


In [0]:
df_inicial = spark.read.option("versionAsOf", version_inicial).table("ventas_delta_demo")

print("Filas iniciales:", df_inicial.count())
print("Filas actuales:", spark.table("ventas_delta_demo").count())
display(df_inicial.filter(F.col("id") == 1))
display(spark.table("ventas_delta_demo").filter(F.col("id") == 1))

Filas iniciales: 100
Filas actuales: 101


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,25.99


id,fecha,producto,cantidad,precio
1,2025-01-03,Teclado,20,27.99


In [0]:
%sql
SELECT 'Pedidos enriquecidos' AS tabla, COUNT(*) AS registros FROM pedidos_enriquecidos
UNION ALL
SELECT 'Ventas base', COUNT(*) FROM ventas
UNION ALL
SELECT 'Ensayo Delta', COUNT(*) FROM ventas_delta_demo

tabla,registros
Pedidos enriquecidos,10
Ventas base,100
Ensayo Delta,101
